# Récupéraion des infos des vagues et de la marée dans le tag d'anguille

Ce notebook utilise des tags d'anguilles avec 2s de périodes d'échantillonage, l'objectif est de calculé **le spectre d'élévation de vagues** à partir des données de pression. 
Pour ensuite comparer avec les spectres d'élévation de vagues de WW3. 

Les étapes sont: 
- 1) Calcul du spectre d'élévation de vagues à partir des données de pression 
- 2) Corriger les niveaux élévation à partir en prenant en compte la profondeur des mesures
- 3) Isoler les fréquences avec de l'information sur les vagues et enlever les fréquences avec uniquement du bruit (périodes associées: 30-12.5s)
- 4) Interpoller les fréquences du spectre de WW3 sur les fréquences du poissons
- 5) Comparer les spectres

In [ ]:
# Set up a local cluster for distributed computing.
from distributed import LocalCluster

cluster = LocalCluster()
client = cluster.get_client()
client

In [ ]:
###########
# Usual libraries
###########
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import dates
import math
import netCDF4 as NC4
import os
import datetime
import pandas as pd
from scipy.fft import rfft, rfftfreq
from scipy.interpolate import interp1d
from scipy.signal import fftconvolve

###########
# Signal processing functions
###########
from scipy.signal import detrend, welch
import xarray as xr

###########
# Additionals functions
###########
from scipy.io import loadmat
from scipy.stats.distributions import chi2
import numpy.matlib
from IPython import display
from scipy.signal import fftconvolve, welch
import hvplot.xarray
import s3fs
import hvplot.xarray  # active le support xarray
import holoviews as hv

## Ouverture WW3 
pas nécessaire pour l'étude spectrale 

**nécessaire pour les pdf de vagues**

In [ ]:
import fsspec
import xarray as xr

# URLs des deux fichiers
urls = [
    "https://data-dataref.ifremer.fr/ww3/resourcecode/HINDCAST/2020/01/FIELD_NC/RSCD_WW3-RSCD-UG_202001_ef.nc",
    "https://data-dataref.ifremer.fr/ww3/resourcecode/HINDCAST/2019/12/FIELD_NC/RSCD_WW3-RSCD-UG_201912_ef.nc",
]
url = "https://data-dataref.ifremer.fr/ww3/resourcecode/HINDCAST/2020/01/FIELD_NC/RSCD_WW3-RSCD-UG_202001_ef.nc"
# --- Ouvrir plusieurs fichiers distants avec fsspec ---
# files = [fsspec.open(url).open() for url in urls]
file = fsspec.open(url).open()
# Charger les deux fichiers dans un seul dataset
ds = xr.open_dataset(file)

# chunks={"node": 1000},  # chunking Dask pour limiter la mémoire
# ou "h5netcdf", parfois plus rapide
# Dask parallélise l'ouverture des fichiers


print(ds)

# Extraire lat / lon
lat = ds["latitude"]
lon = ds["longitude"]

# --- Définir les bornes géographiques ---
lat_min, lat_max = 40, 55
lon_min, lon_max = -10, 5

# Masque sur les nodes
mask_nodes = (lat >= lat_min) & (lat <= lat_max) & (lon >= lon_min) & (lon <= lon_max)

valid_nodes_idx = np.where(mask_nodes.values)[0]  # convertit en indices entiers

# sélectionner ces nodes dans le dataset
WW3 = ds.isel(node=valid_nodes_idx)

WW3

## Définition des fonctions utiles

In [ ]:
from Wave_function import *

**Ouverture du fichier csv avec pandas puis extraction des données avec xarray** 

Le xarray contient: 
- Date (UTC) (dimension)
- Les valeurs de pression (pressure)
- Les valeurs de températures (temperature)
- L'ID du tag (track_tag_id)

Ici il n'y a que les données du tag A15789.

In [ ]:
# On utilise directement les données brutes, donc le csv de base. il se trouve aussi sur le bucket S3 ici: "gfts-ifremer/eel/tag/raw/"

# Ou sur le bucket S3
storage_options = {
    "anon": False,
    "profile": "gfts",
    "client_kwargs": {
        "endpoint_url": "https://s3.gra.perf.cloud.ovh.net",
        "region_name": "gra",
    },
}

fs = s3fs.S3FileSystem(
    anon=storage_options["anon"],
    profile=storage_options["profile"],
    client_kwargs=storage_options["client_kwargs"],
)

tag_name = "A15789"
scratch_root = "s3://gfts-ifremer/eel/raw/"
target_root = f"{scratch_root}"
# data = xr.open_dataset(f"{target_root}/{tag_name}.csv",engine = 'zarr')


# En local
csv_pd_eel = pd.read_csv(
    "/home/jovyan/pangeo-fish/notebooks/nouveau_tag/A15789/dst.csv"
)
# Convertir en xarray Dataset
data = xr.Dataset.from_dataframe(
    csv_pd_eel.set_index(["datetime"])
)  # si tu as une colonne 'tim

data

In [ ]:
import hvplot.xarray

data = data.assign_coords(datetime=pd.to_datetime(data.datetime, dayfirst=True))

# data['pressure'].hvplot()

data_coarse = data.isel(datetime=slice(None, None, 30))

data_coarse["pressure"].hvplot(x="datetime", width=900, height=300)

## Etude spectrale pour identifier la période des vagues
### Correction du spectre d'élévation de vagues à partir des valeurs de profondeurs

On se place lors de périodes de faible activité du poisson, c'est à dire en journée entre 9h et 18h à peu près. 

Il est possible d'adapter la taile de l'échantillon pour éviter de récupérer les **"sauts de profondeurs"** => Passage de 100 mètres à 20 mètres en moisns de 10 minutes. 

De plus, pour visualiser les vagues il faut calculé le spectre sur de grandes échelles temporelles => **Plusieurs heures**. 

In [ ]:
time_sel = "2020-01-20 9:35:00"
date_debut = pd.Timestamp(
    "2019-12-05 14:30:00"
)  # A CHANGER POUR CHAQUE TAG OU UTILISER PREMIER PAS DE TEMPS AUTOMATIQUE

offbot = 0.1  # height of pressure sensor over bottom [m]
zone = 0
fs = 0.5
nfft = 400  # nfft specifies the FFT length that csd uses.
df = fs / nfft  # Frequential resolution
numoverlap = (
    nfft / 2
)  # numoverlap is the number of samples by which the sections overlap.
Nf = int(nfft / 2 + 1)
Nfo = Nf - 1  # up to Nyquist.
# freq=np.linspace(df,df*(Nf-1),Nf-1)

print("Frequency resolution is", df, "Hz.")

# --- Paramètres
fs = 0.5  # Hz (à adapter)
nfft = 400
Nf = int(nfft / 2 + 1)
fmax = 0.08
dfreq = fs / nfft
ifmax = int(fmax / dfreq)


offbot = 0.1  # capteur à 10 cm au-dessus du fond

# --- Extraire un segment
N = int(1 / 2 * 3600 * 5)
date = pd.Timestamp(time_sel)  # "2020-01-27 12:30:00"
indice = conv_time(date_debut, date, 1 / fs)  # fonction du notebook original
elevation = data["pressure"].values[indice : indice + N]
time_p = data["datetime"].values[indice : indice + N]
dep = np.mean(elevation)  # profondeur réelle (m)
print(dep)
# --- Spectral analysis
t = np.arange(len(elevation)) / fs
freq, E_p = welch(
    elevation,
    fs=fs,
    window="hann",
    nperseg=nfft,
    detrend="linear",
    return_onesided=True,
    scaling="density",
)

# --- Définir bande utile (5s - 30s)
f_min = 1 / 30  # ~0.033 Hz
f_max = 1 / 5  # 0.20 Hz
mask_band = (freq >= f_min) & (freq <= f_max)


# --- Inversion dispersion (à coder si pas dispo)
k = dispNewtonTH(freq, dep)
k[0] = 0
M = np.cosh(k * dep) / np.cosh(k * offbot)
plt.plot(freq, M**2)
plt.show()
cosur = M**2
cosur[0] = 1.0
cosur[ifmax + 1 : Nf] = 1.0
Ef = E_p * cosur  # spectre d'élévation corrigé
plt.plot(freq, cosur)
plt.show

# --- Pic spectral
i_peak_raw = np.argmax(E_p[mask_band])
i_peak_corr = np.argmax(E_p[mask_band])
idx_raw = np.where(mask_band)[0][i_peak_raw]
idx_corr = np.where(mask_band)[0][i_peak_corr]

f_peak_raw = freq[idx_raw]
f_peak_corr = freq[idx_corr]
Tp_raw = 1 / f_peak_raw
Tp_corr = 1 / f_peak_corr

# --- Plot
plt.figure(figsize=(12, 4))
plt.plot(t[:N], elevation[:N], "m-")
plt.xlabel("time (s)")
plt.ylabel("pressure (m water)")
plt.title("First 1000 samples of raw pressure data")
plt.show()

plt.figure(figsize=(12, 4))
plt.loglog(freq, E_p, "m-", label="Bottom pressure PSD")
plt.loglog(freq, Ef, "k-", label="Surface elevation PSD (corrected)")
plt.scatter(
    f_peak_corr,
    E_p[idx_corr],
    color="r",
    marker="o",
    s=80,
    label=f"Peak Tp={Tp_corr:.1f}s",
)
plt.xlabel("Frequency (Hz)")
plt.ylabel("PSD (m²/Hz)")
plt.legend()
plt.show()

**E_p = spectre d'élévation à la profonceur z du poisson** 

**Ef spectre corrigé = spectre d'élévation de vagues à la surface.**

In [ ]:
time_sel = pd.Timestamp("2020-01-20 8:10:00")
step_time_plot = 2
N = int(1 / 2 * 3600 * 2)

times_to_plot = [time_sel + pd.Timedelta(hours=step_time_plot * i) for i in range(6)]
print(times_to_plot)
f_min = 1 / 25  # ~0.033 Hz
f_max = 1 / 5  # 0.20 Hz
mask_band = (freq >= f_min) & (freq <= f_max)

# Créer figure avec 6 sous-plots (verticalement)
fig, axes = plt.subplots(nrows=6, ncols=2, figsize=(12, 12))  # 6 lignes, 2 colonnes

for i, date in enumerate(times_to_plot):
    # --- Segment et spectre
    indice = conv_time(date_debut, date, 1 / fs)
    elevation = data["pressure"].values[indice : indice + int(N)]  # / 0.3
    dep = np.mean(elevation)
    t = np.arange(len(elevation)) / fs
    freq, E_p = welch(
        elevation,
        fs=fs,
        window="hann",
        nperseg=nfft,
        detrend="linear",
        return_onesided=True,
        scaling="density",
    )
    # Calcul du Spectre corrigé
    # Inversion dispersion
    # k = dispNewtonTH(freq, dep)
    # k[0] = 0
    # M = np.cosh(k*dep)/np.cosh(k*offbot)
    # cosur = M**2
    # cosur[0] = 1.
    # cosur[ifmax+1:Nf] = 1.
    # Ef = E_p * cosur

    # Calcul du Hs
    iG1 = int(1 / (30 * df))
    iG2 = int(1 / (5 * df))
    Hs = 4 * np.sqrt(np.sum(E_p[iG1:iG2]) * df)
    print("Hs du pas de temps", f"{date.time()}", "est", Hs)
    # Pic spectral
    idx_Ep = np.argmax(E_p[mask_band])
    idx_Ep_global = np.where(mask_band)[0][idx_Ep]
    f_peak_Ep = freq[idx_Ep_global]
    Tp_Ep_corr = 1 / f_peak_Ep

    # --- Plot sur les axes correspondants
    ax_time = axes[i, 0]
    ax_freq = axes[i, 1]
    ax_freq.set_xlim(1 / 30, 1 / 5)
    ax_freq.set_ylim(1e-2, 5e1)

    # Time series
    ax_time.plot(t[:N], elevation[:N], "m-")
    ax_time.set_xlabel("time (s)")
    ax_time.set_ylabel("pressure (m water)")
    ax_time.set_title(f"Raw pressure {date.time()}")

    # Spectre
    ax_freq.loglog(freq, E_p, "m-", label="Bottom pressure PSD")
    ax_freq.scatter(
        f_peak_Ep,
        E_p[idx_Ep_global],
        color="g",
        marker="o",
        s=80,
        label=f"Peak Tp={Tp_Ep_corr:.1f}s\nHs={Hs:.2f} m",
    )
    ax_freq.axvline(f_min, color="gray", ls="--", alpha=0.5)
    ax_freq.axvline(f_max, color="gray", ls="--", alpha=0.5)
    ax_freq.set_xlabel("Frequency (Hz)")
    ax_freq.set_ylabel("PSD (m²/Hz)")
    ax_freq.legend()

plt.tight_layout()
plt.show()

**Reproduction d'une série temporelle d'anguille pour voir comment ça évolue**

In [ ]:
# fs        # fréquence d'échantillonnage
time_len = 2000  # durée du signal (s)
t = np.arange(0, time_len, 1 / fs)

# Périodes (s)
T1 = 18.0
T2 = 11.0
T3 = 8.0
# Pulsations (rad/s)
omega1 = 2 * np.pi / T1
omega2 = 2 * np.pi / T2
omega3 = 2 * np.pi / T3
# Amplitudes
A1 = 2.0
A2 = 1.6
A3 = 0.8

# Reconstruction Signal
signal = (
    120 + A1 * np.cos(omega1 * t) + A2 * np.cos(omega2 * t) + A3 * np.cos(omega3 * t)
)

resolution = 0.3  # m
signal_quantized = np.round(signal * resolution)  # / resolution

# ______________________________________________________________________________________________________________________________________
### Histogramme sur les données réelles ###
# ______________________________________________________________________________________________________________________________________

date_hist = pd.Timestamp("2020-01-22 10:10:00")
indice = conv_time(date_debut, date_hist, 1 / fs)
elevation = data["pressure"].values[indice : indice + int(time_len * fs)]

# Plot reconstruction
plt.figure(figsize=(5, 2))
# plt.plot(t,signal)
plt.plot(t, signal_quantized)
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title("Signal cosinus à deux composantes (T=18s et T=12s)")
plt.grid(True)
plt.show()

# Plot série temporelle
plt.figure(figsize=(5, 2))
# plt.plot(t,signal)
plt.plot(t, elevation)
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title("Signal cosinus à deux composantes (T=18s et T=12s)")
plt.grid(True)
plt.show()

# Histogramme reconstruction
plt.figure(figsize=(5, 2))
plt.hist(signal_quantized, bins=50, edgecolor="black", alpha=0.75)
plt.xlabel("Amplitude (cm)")
plt.ylabel("Nombre d'occurrences")
plt.title("Histogramme du signal quantifié (résolution 30 cm)")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

# Histogramme série temporelle
plt.figure(figsize=(5, 2))
plt.hist(elevation, bins=50, edgecolor="black", alpha=0.75)
plt.xlabel("Amplitude (cm)")
plt.ylabel("Nombre d'occurrences")
plt.title("Histogramme du série temporelle de pression")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# --- Définir bande utile (5s - 30s)
f_min = 1 / 30  # ~0.033 Hz
f_max = 1 / 5  # 0.20 Hz
mask_band = (freq >= f_min) & (freq <= f_max)

# --- Pic spectral
i_peak_raw = np.argmax(E_p[mask_band])
i_peak_corr = np.argmax(Ef[mask_band])
idx_raw = np.where(mask_band)[0][i_peak_raw]
idx_corr = np.where(mask_band)[0][i_peak_corr]

f_peak_raw = freq[idx_raw]
f_peak_corr = freq[idx_corr]
Tp_raw = 1 / f_peak_raw
Tp_corr = 1 / f_peak_corr

# --- Moments spectraux pour Tm
df = freq[1] - freq[0]  # résolution fréquentielle
m0 = np.sum(Ef[mask_band]) * df
m1 = np.sum(freq[mask_band] * Ef[mask_band]) * df
Tm01 = m0 / m1

print("=== Paramètres spectraux (bande 5–30 s) ===")
print(
    f"Pic (raw) : Tp = {Tp_raw:.1f} s à f = {f_peak_raw:.4f} Hz (PSD={E_p[idx_raw]:.2e})"
)
print(
    f"Pic (corr) : Tp = {Tp_corr:.1f} s à f = {f_peak_corr:.4f} Hz (PSD={Ef[idx_corr]:.2e})"
)
print(f"Période moyenne spectrale Tm01 = {Tm01:.1f} s")
# --- Plot
plt.figure(figsize=(12, 4))
plt.plot(t[:N], elevation[:N], "m-")
plt.xlabel("time (s)")
plt.ylabel("pressure (m water)")
plt.title("First 1000 samples of raw pressure data")
plt.show()
# --- Visualisation
plt.figure(figsize=(12, 4))
plt.loglog(freq, E_p, "m-", label="Bottom pressure PSD")
plt.loglog(freq, Ef, "k-", label="Surface elevation PSD (corrected)")
plt.scatter(
    f_peak_corr,
    Ef[idx_corr],
    color="r",
    marker="o",
    s=80,
    label=f"Peak Tp={Tp_corr:.1f}s",
)
# plt.axvline(1/Tm01, color="b", ls="--", label=f"Tm01={Tm01:.1f}s")
plt.axvline(f_min, color="gray", ls="--", alpha=0.5, label="5–30s band")
plt.axvline(f_max, color="gray", ls="--", alpha=0.5)
plt.xlabel("Frequency (Hz)")
plt.ylabel("PSD (m²/Hz)")
plt.legend()
plt.title("Spectral parameters: Tp and Tm01 (5–30s)")
plt.show()

iG1 = int(1 / (30 * df))
iG2 = int(1 / (5 * df))
Hs = 4 * np.sqrt(np.sum(Ef[iG1:iG2]) * df)
print("Le Hs calculé sur l'élévation corrigé pour ce segment est:", Hs)

In [ ]:
freq_min = 1 / 30  # En lien avec les valeurs de WW3
freq_max = 0.25  # ≈12.5 s, au-delà on est dans le bruit

# --- Masque de fréquence pour le spectre de l'anguille
mask_freq = (freq >= freq_min) & (freq <= freq_max)
E_p_filtered = E_p[mask_freq]
Ef_filtered = Ef[mask_freq]
freq_filtered = freq[mask_freq]

# --- Récupération des deux spectres WW3
node = station_proche(WW3, 50.035443, -3.586503)

# Premier spectre WW3
ef_log_1 = node["ef"].sel(time=pd.Timestamp(time_sel), method="nearest").values
ef_1 = np.power(10, ef_log_1) - 1e-12  # conversion log -> linéaire

# Deuxième spectre WW3
ef_log_2 = (
    WW3["ef"].isel(node=82560).sel(time=pd.Timestamp(time_sel), method="nearest").values
)
ef_2 = np.power(10, ef_log_2) - 1e-12  # conversion log -> linéaire

# Fréquences de WW3
f = node["f"].values

# --- Interpolation des deux spectres WW3 sur la grille du spectre anguille
da_ef1 = xr.DataArray(
    data=ef_1, coords={"f": f, "time": pd.Timestamp(time_sel)}, dims=["f"], name="ef"
)
ds_ef1 = xr.Dataset({"ef": da_ef1})
newds_ef1 = bin_over_f(ds_ef1, freq_filtered)
ef_WW3_1 = newds_ef1["ef"].values

da_ef2 = xr.DataArray(
    data=ef_2, coords={"f": f, "time": pd.Timestamp(time_sel)}, dims=["f"], name="ef"
)
ds_ef2 = xr.Dataset({"ef": da_ef2})
newds_ef2 = bin_over_f(ds_ef2, freq_filtered)
ef_WW3_2 = newds_ef2["ef"].values

# --- Plot
plt.figure(figsize=(12, 5))
plt.loglog(freq_filtered, Ef_filtered, "m-", label="Eel PSD (corrected)")
plt.loglog(freq_filtered, ef_WW3_1, "b-", label="WW3 PSD – estimation PDF")
plt.loglog(freq_filtered, ef_WW3_2, "g-", label="WW3 PSD – via géoloc eel")

# Options supplémentaires
plt.axvline(freq_min, color="gray", ls="--", alpha=0.5, label="12–30 s band")
plt.axvline(freq_max, color="gray", ls="--", alpha=0.5)
plt.grid(True, which="both", ls="--", alpha=0.3)
plt.xlabel("Frequency (Hz)")
plt.ylabel("PSD (m²/Hz)")
plt.legend()
plt.title("Comparison of Eel PSD with Two WW3 Spectra")
plt.tight_layout()
plt.show()

### Utilisation de la différence par norme, cela ne marche pas.  

In [ ]:
# Diff calcul de la différence pour un segment de donnée brute d'anguille avec WW3
diff = selection_WW3(pd.Timestamp(time_sel), Ef_filtered, freq_filtered, WW3)
# On rajoute la dimension time dans le dataset
diff_expanded = diff.expand_dims(time=[diff.time.values])

diff_sum = xr.DataArray(
    np.where(
        np.all(np.isnan(diff_expanded), axis=1),
        np.nan,
        (np.nansum(diff_expanded, axis=1)),
    ),
    dims=["time", "node"],
    coords={"time": diff_expanded.time, "node": diff_expanded.node},
)

#  Dataset regroupant diff et sa somme quadratique
ds_diff = xr.Dataset({"diff": diff_expanded, "diff_sum_quad": diff_sum})

# Ajout des coordonnées géographiques
ds_diff = ds_diff.assign_coords(
    {
        "latitude": ("node", WW3.latitude.values),
        "longitude": ("node", WW3.longitude.values),
    }
)

# 4) Extraction du DataArray pour un instant donné (time=0 dans ton exemple)
ds_node = ds_diff.diff_sum_quad.isel(time=0)

# 5) Trouver les indices des 5 plus petites valeurs en ignorant les NaN
values = ds_node.values  # shape = (node,)
# Crée un masque pour ignorer les NaN
mask = ~np.isnan(values)
top5_indices = np.argsort(values[mask])[:5]  # indices parmi les valeurs valides
# Récupère les indices originaux dans ds_node
top5_nodes = np.arange(len(values))[mask][top5_indices]
top5_values = values[top5_nodes]

# 6) Affichage
print("Indices des 5 plus petits nodes :", top5_nodes)
print("Valeurs correspondantes :", top5_values)
# On peut aussi récupérer latitude/longitude associées
print("Latitudes :", ds_diff.latitude.values[top5_nodes])
print("Longitudes :", ds_diff.longitude.values[top5_nodes])


import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import xarray as xr

# -------------------------------
# 0) Sélection du temps
# -------------------------------
# On prend par exemple le premier instant
t_index = 0
dist_sel = ds_diff["diff_sum_quad"].isel(time=t_index)

# Extraction des lat/lon pour les mêmes nodes
lat_sel = ds_diff["latitude"]
lon_sel = ds_diff["longitude"]

# Conversion en array pour le calcul des stats
dist_values = dist_sel.values

# -------------------------------
# 1) PDF à partir des valeurs
# -------------------------------
# On retire les NaN pour le calcul du PDF
dist_valid = dist_values[~np.isnan(dist_values)]

# Histogramme normalisé
counts, bins = np.histogram(dist_valid, bins=50, density=True)
bin_centers = 0.5 * (bins[1:] + bins[:-1])

# Estimation KDE
kde = gaussian_kde(dist_valid)
x_eval = np.linspace(dist_valid.min(), dist_valid.max(), 200)
pdf_vals = kde(x_eval)

# -------------------------------
# 2) Définir les limites géographiques pour la carte
# -------------------------------
lon_min, lon_max = np.nanmin(lon_sel), np.nanmax(lon_sel)
lat_min, lat_max = np.nanmin(lat_sel), np.nanmax(lat_sel)

# -------------------------------
# 3) Visualisation
# -------------------------------
fig = plt.figure(figsize=(14, 6))

# a) Distribution PDF
ax1 = fig.add_subplot(1, 2, 1)
ax1.plot(x_eval, pdf_vals, "r-", label="KDE PDF")
ax1.hist(dist_valid, bins=50, density=True, alpha=0.3, label="Histogramme")
ax1.set_xlabel("Distance spectrale quadratique")
ax1.set_ylabel("Densité de probabilité")
ax1.legend()
ax1.set_title(f"Distribution des distances (t={str(dist_sel.time.values)[:19]})")

# b) Carte géographique
ax2 = fig.add_subplot(1, 2, 2, projection=ccrs.PlateCarree())
sc = ax2.scatter(
    lon_sel, lat_sel, c=dist_values, s=5, cmap="jet_r", transform=ccrs.PlateCarree()
)
ax2.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
ax2.coastlines(resolution="10m")
ax2.add_feature(cfeature.LAND, facecolor="lightgray")
ax2.gridlines(draw_labels=True)

cbar = plt.colorbar(sc, ax=ax2, orientation="vertical", label="Distance spectrale")
ax2.set_title("Carte des distances spectrales (WW3 vs référence)")

plt.tight_layout()
plt.show()

### Utilisation d'une norme euclidienne normalisée

In [ ]:
ef_ref_bottom = xr.DataArray(E_p_filtered, dims="f", coords={"f": freq_filtered})
ef_ref = xr.DataArray(Ef_filtered, dims="f", coords={"f": freq_filtered})
diff_t = diff_expanded.isel(time=0)  # (f, node)

norm_ref = np.sqrt(np.nansum(ef_ref.values**2))


vals = diff_t.values  # (n_freq, n_node)

# calcul de la distance en ignorant les NaN
# (on somme seulement sur les fréquences valides)
somme = np.nansum((vals / norm_ref) ** 2, axis=0)

# détection des nodes sans aucune valeur valide
all_nan_mask = np.all(np.isnan(vals), axis=0)

# on met la distance à +inf pour ces nodes
somme[all_nan_mask] = np.inf

dist_nodes = np.sqrt(somme)

print("Shape dist_nodes :", dist_nodes.shape)
print("Nb de nodes sans données :", np.sum(np.isinf(dist_nodes)))

# récupération des 5 plus proches
top5_indices = np.argsort(dist_nodes)[:5]
top5_values = dist_nodes[top5_indices]

print("Indices des 5 plus proches :", top5_indices)
print("Distances correspondantes :", top5_values)

print("Latitudes :", ds_diff.latitude.values[top5_indices])
print("Longitudes :", ds_diff.longitude.values[top5_indices])

# Zone d’étude
lat_min, lat_max = 40, 55
lon_min, lon_max = -10, 5

# Extraction des lat/lon et distances
lat = ds_diff.latitude.values
lon = ds_diff.longitude.values
dist = dist_nodes.copy()

# Filtrage de la zone et des valeurs valides
mask = (
    (lat >= lat_min)
    & (lat <= lat_max)
    & (lon >= lon_min)
    & (lon <= lon_max)
    & np.isfinite(dist)
)
lat_sel = lat[mask]
lon_sel = lon[mask]
dist_sel = dist[mask]

print(f"Nombre de nœuds sélectionnés : {len(dist_sel)}")

# -------------------------------
# 1) PDF à partir des valeurs
# -------------------------------
# Histogramme normalisé
counts, bins = np.histogram(dist_sel, bins=50, density=True)
bin_centers = 0.5 * (bins[1:] + bins[:-1])

# Estimation lissée (KDE)
kde = gaussian_kde(dist_sel)
x_eval = np.linspace(dist_sel.min(), dist_sel.max(), 200)
pdf_vals = kde(x_eval)

# -------------------------------
# 2) Visualisation
# -------------------------------

fig = plt.figure(figsize=(12, 6))

# a) PDF
ax1 = fig.add_subplot(1, 2, 1)
ax1.plot(x_eval, pdf_vals, label="KDE PDF")
ax1.hist(dist_sel, bins=50, density=True, alpha=0.3, label="Histogramme")
ax1.set_xlabel("Distance spectrale normalisée")
ax1.set_ylabel("Densité de probabilité")
ax1.legend()
ax1.set_title("Distribution des distances")

# b) Carte
ax2 = fig.add_subplot(1, 2, 2, projection=ccrs.PlateCarree())
sc = ax2.scatter(
    lon_sel, lat_sel, c=dist_sel, s=5, cmap="jet_r", transform=ccrs.PlateCarree()
)
ax2.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
ax2.coastlines(resolution="10m")
ax2.add_feature(cfeature.LAND, facecolor="lightgray")
ax2.gridlines(draw_labels=True)
cbar = plt.colorbar(sc, ax=ax2, orientation="vertical", label="Distance spectrale")

ax2.set_title("Carte des distances (WW3 vs référence)")

plt.tight_layout()
plt.show()

### Test indicateur chatGTP

In [ ]:
# ef_model_da : model ef (time,f,node) (xarray)
# ef_ref_da   : Ef_filtered (f,) (xarray)

ds_slice = WW3.sel(time=pd.Timestamp(time_sel), method="nearest")
ds_interp = ds_slice.interp(f=freq_filtered)
metrics_ds = compute_spectral_similarity(
    ef_model_da=ds_interp["ef"], ef_ref_da=ef_ref, time=time_sel
)
print(metrics_ds)

import numpy as np

# top 10 par cosine
topk = 10
valid = np.isfinite(metrics_ds["cosine"].values)
top_cos = np.argsort(-metrics_ds["cosine"].values[valid])[:topk]  # descending
nodes_valid = metrics_ds["cosine"].node.values[valid]
top_nodes_cos = nodes_valid[top_cos]
print("Top cosine nodes:", top_nodes_cos)

# top 10 par JS (smallest)
valid_js = np.isfinite(metrics_ds["JS_sqrt"].values)
top_js = np.argsort(metrics_ds["JS_sqrt"].values[valid_js])[:topk]
nodes_valid_js = metrics_ds["JS_sqrt"].node.values[valid_js]
top_nodes_js = nodes_valid_js[top_js]
print("Top JS nodes:", top_nodes_js)

# top 10 par JS (smallest)
valid_log = np.isfinite(metrics_ds["spectral_angle"].values)
top_log = np.argsort(metrics_ds["spectral_angle"].values[valid_log])[:topk]
nodes_valid_log = metrics_ds["spectral_angle"].node.values[valid_log]
top_nodes_log = nodes_valid_log[top_log]
print("Top spectral_angle nodes:", top_nodes_log)

top_latitudes = WW3.latitude.values[top_nodes_log]
top_longitudes = WW3.longitude.values[top_nodes_log]

# Affichage
for i, node in enumerate(top_nodes_log):
    print(
        f"Node {node}: latitude = {top_latitudes[i]}, longitude = {top_longitudes[i]}"
    )


# Sélection des valeurs WRMSE et des coordonnées correspondantes
wrmse_vals = metrics_ds[
    "pearson"
].values  # shape (node,)    <<<<<<<<-------------------------------------------------------------------------
lat_vals = WW3.latitude.values
lon_vals = WW3.longitude.values

# Filtrer sur la zone d'étude et valeurs finies
mask = (
    (lat_vals >= lat_min)
    & (lat_vals <= lat_max)
    & (lon_vals >= lon_min)
    & (lon_vals <= lon_max)
    & np.isfinite(wrmse_vals)
)

wrmse_sel = wrmse_vals[mask]
lat_sel = lat_vals[mask]
lon_sel = lon_vals[mask]

# -------------------------------
# 1) PDF à partir des valeurs
# -------------------------------
counts, bins = np.histogram(wrmse_sel, bins=50, density=True)
bin_centers = 0.5 * (bins[1:] + bins[:-1])

# Estimation lissée (KDE)
kde = gaussian_kde(wrmse_sel)
x_eval = np.linspace(wrmse_sel.min(), wrmse_sel.max(), 200)
pdf_vals = kde(x_eval)

# -------------------------------
# 2) Visualisation
# -------------------------------
fig = plt.figure(figsize=(12, 6))

# a) PDF
ax1 = fig.add_subplot(1, 2, 1)
ax1.plot(x_eval, pdf_vals, label="KDE PDF", color="blue")
ax1.hist(
    wrmse_sel, bins=50, density=True, alpha=0.3, label="Histogramme", color="skyblue"
)
ax1.set_xlabel("pearson")
ax1.set_ylabel("Densité de probabilité")
ax1.legend()
ax1.set_title("Distribution des spectral_angle")

# b) Carte
ax2 = fig.add_subplot(1, 2, 2, projection=ccrs.PlateCarree())

# Colormap inversée pour que les plus petites valeurs soient plus visibles
sc = ax2.scatter(
    lon_sel,
    lat_sel,
    c=wrmse_sel,
    s=10,
    cmap="jet_r",  # inversé
    transform=ccrs.PlateCarree(),
    vmin=np.nanmin(wrmse_sel),  # optionnel pour contrôle contraste
    vmax=np.nanpercentile(wrmse_sel, 95),  # saturation des outliers
)

ax2.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
ax2.coastlines(resolution="10m")
ax2.add_feature(cfeature.LAND, facecolor="lightgray")
ax2.gridlines(draw_labels=True)
cbar = plt.colorbar(sc, ax=ax2, orientation="vertical", label="spectral_angle")
ax2.set_title("Carte des spectral_angle ")

plt.tight_layout()

# Sélection des valeurs WRMSE et des coordonnées correspondantes
cosine_vals = metrics_ds["cosine"].values  # shape (node,)
lat_vals = WW3.latitude.values
lon_vals = WW3.longitude.values

# Filtrer sur la zone d'étude et valeurs finies
mask = (
    (lat_vals >= lat_min)
    & (lat_vals <= lat_max)
    & (lon_vals >= lon_min)
    & (lon_vals <= lon_max)
    & np.isfinite(cosine_vals)
)

cosine_sel = cosine_vals[mask]
lat_sel = lat_vals[mask]
lon_sel = lon_vals[mask]

# -------------------------------
# 1) PDF à partir des valeurs
# -------------------------------
counts_c, bins_c = np.histogram(cosine_sel, bins=50, density=True)
bin_centers_c = 0.5 * (bins_c[1:] + bins_c[:-1])

# Estimation lissée (KDE)
kde_c = gaussian_kde(cosine_sel)
x_eval_c = np.linspace(cosine_sel.min(), cosine_sel.max(), 200)
pdf_vals_c = kde_c(x_eval_c)

# -------------------------------
# 2) Visualisation
# -------------------------------
fig = plt.figure(figsize=(12, 6))

# a) PDF
ax1_c = fig.add_subplot(1, 2, 1)
ax1_c.plot(x_eval_c, pdf_vals_c, label="KDE PDF", color="blue")
ax1_c.hist(
    cosine_sel, bins=50, density=True, alpha=0.3, label="Histogramme", color="skyblue"
)
ax1_c.set_xlabel("pearson")
ax1_c.set_ylabel("Densité de probabilité")
ax1_c.legend()
ax1_c.set_title("Distribution des cosine")

# b) Carte
ax2_c = fig.add_subplot(1, 2, 2, projection=ccrs.PlateCarree())

# Colormap inversée pour que les plus petites valeurs soient plus visibles
sc_c = ax2_c.scatter(
    lon_sel,
    lat_sel,
    c=cosine_sel,
    s=10,
    cmap="jet",  # inversé
    transform=ccrs.PlateCarree(),
    vmin=np.nanmin(cosine_sel),  # optionnel pour contrôle contraste
    vmax=np.nanpercentile(cosine_sel, 95),  # saturation des outliers
)

ax2_c.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
ax2_c.coastlines(resolution="10m")
ax2_c.add_feature(cfeature.LAND, facecolor="lightgray")
ax2_c.gridlines(draw_labels=True)
cbar = plt.colorbar(sc_c, ax=ax2_c, orientation="vertical", label="cosine")
ax2_c.set_title("Carte des cosine ")

plt.tight_layout()

plt.show()

___________________________________________________________________________________________

## Test détection des périodes de calmes

In [ ]:
# utilisation de la fonction
calm_periods = detect_calm_periods(
    data["pressure"],
    fs=0.5,  # fréquence d’échantillonnage (Hz)
    var_window_minutes=5,  # taille de la fenêtre pour calculer la variance
    calm_var_threshold=10.0,  # seuil de variance pour considérer "calme"
    min_calm_duration_minutes=120,  # durée minimale pour valider une période calme
)
print(len(calm_periods))
import matplotlib.pyplot as plt

# Extraction du signal et du temps
signal = data["pressure"].values
time_index = data["datetime"].values  # vecteur datetime64

# Sélection des 5 périodes calmes
examples = calm_periods[40:44]
print("ok")
fig, axes = plt.subplots(len(examples), 1, figsize=(14, 12), sharex=False)

if len(examples) == 1:
    axes = [axes]  # en cas d'un seul subplot

for ax, period in zip(axes, examples):
    s, e = period["start_idx"], period["end_idx"]

    # marge avant/après pour le contexte
    margin = 500
    start_plot = max(s - margin, 0)
    end_plot = min(e + margin, len(signal))

    # tracer le signal avec le temps en abscisse
    ax.plot(
        time_index[start_plot:end_plot],
        signal[start_plot:end_plot],
        color="steelblue",
        lw=1,
        label="Signal",
    )

    # surligner la période calme
    ax.axvspan(
        time_index[s], time_index[e], color="orange", alpha=0.3, label="Période calme"
    )

    # lignes verticales
    ax.axvline(time_index[s], color="red", linestyle="--", lw=1)
    ax.axvline(time_index[e], color="red", linestyle="--", lw=1)

    # titre
    ax.set_title(
        f"Période calme | Début: {time_index[s]}  "
        f"Durée: {period['duration_s']:.0f} s"
    )

    ax.set_ylabel("Pression")
    print("ok boucle")
axes[-1].set_xlabel("Temps (datetime)")

fig.autofmt_xdate()  # incliner les labels de dates pour lisibilité
plt.tight_layout()
plt.show()

# Recherches divers 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import welch


def process_calm_segment(
    segment_info, data, fs=0.5, offbot=0.1, dep=None, nfft=400, fmax_plot=0.08
):
    """
    segment_info : dict avec 'start_idx', 'end_idx', 'duration_s'
    data : xarray Dataset avec variable 'pressure' et coord 'datetime'
    fs : fréquence d'échantillonnage (Hz)
    offbot : hauteur du capteur au-dessus du fond (m)
    dep : profondeur moyenne (m), si None, calculée à partir de segment
    nfft : longueur FFT
    fmax_plot : fréquence max pour la correction M²
    """
    start_idx = segment_info["start_idx"]
    end_idx = segment_info["end_idx"]
    N = end_idx - start_idx + 1

    elevation = data["pressure"].values[start_idx : end_idx + 1]
    time_p = data["datetime"].values[start_idx : end_idx + 1]

    if dep is None:
        dep = np.mean(elevation)
    print(f"Profondeur moyenne pour le segment : {dep:.2f} m")

    # --- Spectre
    freq, E_p = welch(
        elevation,
        fs=fs,
        window="hann",
        nperseg=nfft,
        detrend="linear",
        return_onesided=True,
        scaling="density",
    )

    # --- Inversion dispersion
    k = dispNewtonTH(freq, dep)  # à définir par l'utilisateur
    k[0] = 0
    M = np.cosh(k * dep) / np.cosh(k * offbot)
    cosur = M**2
    cosur[0] = 1.0

    # fréquence maximale pour correction
    Nf = len(freq)
    ifmax = np.searchsorted(freq, fmax_plot)
    cosur[ifmax + 1 : Nf] = 1.0

    # --- Plot M²
    plt.figure(figsize=(10, 3))
    plt.plot(freq, cosur)
    plt.xlabel("Frequency (Hz)")
    plt.ylabel("M² correction factor")
    plt.title("Correction for surface elevation (M²)")
    plt.show()

    # --- Spectre corrigé
    Ef = E_p * cosur

    # --- Bande utile
    f_min, f_max = 1 / 30, 1 / 5
    mask_band = (freq >= f_min) & (freq <= f_max)

    # --- Pic spectral
    idx_corr = np.argmax(Ef[mask_band])
    idx_Ep = np.argmax(E_p[mask_band])
    idx_corr_global = np.where(mask_band)[0][idx_corr]
    idx_Ep_global = np.where(mask_band)[0][idx_Ep]
    f_peak_corr = freq[idx_corr_global]
    f_peak_Ep = freq[idx_Ep_global]
    Tp_corr = 1 / f_peak_corr
    Tp_Ep_corr = 1 / f_peak_Ep

    # --- Moments spectraux
    dfreq = freq[1] - freq[0]
    m0 = np.sum(Ef[mask_band]) * dfreq
    m1 = np.sum(freq[mask_band] * Ef[mask_band]) * dfreq
    Tm01 = m0 / m1

    # --- Hs
    iG1, iG2 = np.searchsorted(freq, [f_min, f_max])
    Hs = 4 * np.sqrt(np.sum(Ef[iG1:iG2]) * dfreq)

    # --- Plot série temporelle
    plt.figure(figsize=(12, 4))
    plt.plot(time_p, elevation, "m-")
    plt.xlabel("Time")
    plt.ylabel("Pressure (m water)")
    plt.title(f"Segment du {time_p[0]} au {time_p[-1]}")
    plt.show()

    # --- Plot PSD
    plt.figure(figsize=(12, 4))
    plt.loglog(freq, E_p, "m-", label="Bottom pressure PSD")
    plt.loglog(freq, Ef, "k-", label="Surface elevation PSD (corrected)")
    plt.scatter(
        f_peak_corr,
        Ef[idx_corr_global],
        color="r",
        marker="o",
        s=80,
        label=f"Peak corrigée Tp={Tp_corr:.1f}s",
    )
    plt.scatter(
        f_peak_Ep,
        E_p[idx_Ep_global],
        color="g",
        marker="o",
        s=80,
        label=f"Peak Tp={Tp_Ep_corr:.1f}s",
    )
    plt.axvline(f_min, color="gray", ls="--", alpha=0.5, label="5–30s band")
    plt.axvline(f_max, color="gray", ls="--", alpha=0.5)
    plt.xlabel("Frequency (Hz)")
    plt.ylabel("PSD (m²/Hz)")
    plt.legend()
    plt.title(f"Spectral parameters: Tp and Tm01 (5–30s)")
    plt.show()

    print(f"=== Résultats spectraux ===")
    print(f"Pic corrigé : Tp = {Tp_corr:.1f} s à f = {f_peak_corr:.4f} Hz")
    print(f"Période moyenne Tm01 = {Tm01:.1f} s")
    print(f"Hauteur significative Hs = {Hs:.2f} m")

    return {
        "freq": freq,
        "E_p": E_p,
        "Ef": Ef,
        "Tp_corr": Tp_corr,
        "Tm01": Tm01,
        "Hs": Hs,
        "M2": cosur,
    }


# --- Exemple d'utilisation
list_seg = calm_periods[40:44]
for segment_info in list_seg:
    results = process_calm_segment(segment_info, data)

In [ ]:
# Conversion indices -> timestamps
time_start = pd.Timestamp(data["datetime"].values[segment_info["start_idx"]])
time_end = pd.Timestamp(data["datetime"].values[segment_info["end_idx"]])

print("Plage temporelle de la période calme :", time_start, " -> ", time_end)


# On sélectionne uniquement les pas horaires de WW3 qui tombent dans cette plage
WW3_seg = WW3.sel(time=slice(time_start, time_end))

WW3_interp = WW3_seg.interp(f=freq_filtered)

In [ ]:
def crea_dataarray(Ef, freq, f_min=1 / 30, f_max=0.25):
    mask_freq = (freq >= f_min) & (freq <= f_max)
    Ef_filtered = Ef[mask_freq]
    freq_filtered = freq[mask_freq]
    return xr.DataArray(Ef_filtered, dims="f", coords={"f": freq_filtered})


# retourne un dataArray avec l'élévation spectrale corrigé de l'anguille

In [ ]:
result_list = []
N = len(WW3_interp["time"])
time = WW3_interp["time"].values

ef_ref_eel = crea_dataarray(results["E_p"], results["freq"], f_min=1 / 30, f_max=0.25)


for i, t in enumerate(time):
    print(t)
    ds_i = compute_spectral_similarity(
        ef_model_da=WW3_interp["ef"].isel(time=i),
        ef_ref_da=ef_ref_eel,
        time=t,
        time_index=i,
        eps=1e-12,
    )
    result_list.append(ds_i)
calcul_all = xr.concat(result_list, dim="time")
print(calcul_all)

In [ ]:
ef_ref_eel.plot(xscale="log", yscale="log")
calcul_all["time"].values

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# ----------------------------
# Dataset complet déjà calculé
# calcul_all : Dataset
# dims: (time, node)
# variables: cosine, spectral_angle, JS_sqrt, ...
# ----------------------------

times = calcul_all["time"].values
lat_vals = WW3.latitude.values
lon_vals = WW3.longitude.values
print(times)
# --- Boucle sur chaque pas de temps
for it, t in enumerate(times):

    # Extraire les vecteurs des métriques au temps it
    cosine_vals = calcul_all["cosine"].isel(time=it).values
    spectral_angle_vals = calcul_all["spectral_angle"].isel(time=it).values
    JS_vals = calcul_all["JS_sqrt"].isel(time=it).values

    metrics_dict = {
        "spectral_angle": spectral_angle_vals,
        "wrmse": cosine_vals,
        "JS_sqrt": JS_vals,
    }

    print(f"=== Temps {t} ===")

    for key, vals in metrics_dict.items():
        # --- Masque spatial et valeurs finies
        mask = (
            (lat_vals >= lat_min)
            & (lat_vals <= lat_max)
            & (lon_vals >= lon_min)
            & (lon_vals <= lon_max)
            & np.isfinite(vals)
        )

        vals_sel = vals[mask]
        lat_sel = lat_vals[mask]
        lon_sel = lon_vals[mask]

        if len(vals_sel) < 10:
            print(f"[WARN] Peu de données valides pour {key} au temps {t}")
            continue

        # ===========================
        # PDF avec KDE
        # ===========================
        kde = gaussian_kde(vals_sel)
        x_eval = np.linspace(vals_sel.min(), vals_sel.max(), 200)
        pdf_vals = kde(x_eval)

        # ===========================
        # FIGURE
        # ===========================
        fig = plt.figure(figsize=(12, 6))

        # a) PDF
        ax1 = fig.add_subplot(1, 2, 1)
        ax1.plot(x_eval, pdf_vals, label="KDE PDF", color="blue")
        ax1.hist(
            vals_sel,
            bins=50,
            density=True,
            alpha=0.3,
            label="Histogramme",
            color="skyblue",
        )
        ax1.set_xlabel(key)
        ax1.set_ylabel("Densité de probabilité")
        ax1.legend()
        ax1.set_title(f"Distribution {key} – {times[it]}")

        # b) Carte
        ax2 = fig.add_subplot(1, 2, 2, projection=ccrs.PlateCarree())
        sc = ax2.scatter(
            lon_sel,
            lat_sel,
            c=vals_sel,
            s=10,
            cmap="jet" if key == "cosine" else "jet_r",  # couleurs
            transform=ccrs.PlateCarree(),
            vmin=np.nanmin(vals_sel),
            vmax=np.nanpercentile(vals_sel, 95),  # saturer les outliers
        )
        ax2.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
        ax2.coastlines(resolution="10m")
        ax2.add_feature(cfeature.LAND, facecolor="lightgray")
        ax2.gridlines(draw_labels=True)
        plt.colorbar(sc, ax=ax2, orientation="vertical", label=key)
        ax2.set_title(f"Carte {key} – {times[i]}")

        plt.tight_layout()
        plt.show()

## Alignement et suppression des sauts pour la marée

### Etude de la variance pour localiser les sauts dans les données. 

In [ ]:
# --- Extraire un segment
N = 50000

date2 = pd.Timestamp("2020-01-20 13:00:00")
indice = conv_time(date_debut, date2, 1 / fs)  # fonction du notebook original
segment = data["pressure"].values[indice : indice + N]
time_p = data["datetime"].values[indice : indice + N]
print(segment)
# On suppose que 'segment' contient les valeurs de pression
# et 'time_p' les timestamps correspondants (dtype=datetime64)


# On suppose que 'segment' contient les valeurs de pression
# et 'time_p' les timestamps correspondants (dtype=datetime64)

# Construire une série pandas avec un index temporel
series = pd.Series(segment, index=pd.to_datetime(time_p))

# Calcul de la variance par fenêtre de 5 minutes
var_5min = (
    data.pressure.isel(datetime=slice(indice, indice + N))
    .rolling(datetime=int(5 * 60 * 0.5), center=True)
    .var()
)

print(var_5min)


# Tracer la variance
# plt.figure(figsize=(10,5))
# plt.plot(var_5min.index, var_5min.values, marker='o')
# plt.title("Variance du signal de pression (fenêtres de 5 min)")
# plt.xlabel("Temps")
# plt.ylabel("Variance")
# plt.grid(True)
# plt.show()


data.pressure.isel(datetime=slice(indice, indice + N)).hvplot()

In [ ]:
fs = 0.5  # Hz
window_points = int(5 * 60 * fs)  # = 150 points
segment_da = data.pressure.isel(datetime=slice(indice, indice + N))
var_5min = segment_da.rolling(datetime=window_points, center=True).var()
step_points = int(60 * fs)  # = 60 points si fs=0.5 Hz
# convertir DataArray en array numpy pour le calcul
var_array = var_5min.values
n_windows = len(var_array) // step_points

mean_var_2min = np.array(
    [
        var_array[i * step_points : (i + 1) * step_points].mean()
        for i in range(n_windows)
    ]
)


time_2min = np.array(
    [segment_da.isel(datetime=i * step_points).values for i in range(n_windows)]
)

plt.figure(figsize=(10, 5))
# plt.plot(segment_da.values, label="Signal original")
plt.plot(
    np.arange(len(mean_var_2min)) * step_points,
    mean_var_2min,
    "o-",
    label="Variance 5min moyenne 2min",
)
plt.legend()
plt.show()

In [ ]:
import hvplot.pandas  # ou hvplot.xarray si tu utilises xarray

# Créer le graphique log-log

# Convertir tes données en DataFrame pour hvPlot
df = pd.DataFrame({"freq": freq, "E_p": E_p, "Ef": Ef})

plot = df.hvplot.line(
    x="freq",
    y="E_p",
    logx=True,
    logy=True,
    color="magenta",
    label="Bottom pressure PSD",
    width=800,
    height=300,
) * df.hvplot.line(
    x="freq",
    y="Ef",
    logx=True,
    logy=True,
    color="black",
    label="Surface elevation PSD (corrected)",
)

# Ajouter labels et légende
plot = plot.opts(
    xlabel="Frequency (Hz)",
    ylabel="PSD (m²/Hz)",
    legend_position="top_right",
    title="PSD from bottom pressure vs corrected surface elevation",
)

plot

In [ ]:
# --- Paramètres ---
time_sel = "2020-01-23 8:35:00"
offbot = 0.1
fs = 0.5
nfft = 400
f_min = 1 / 25  # 0.04 Hz  <=> 25 s
f_max = 1 / 5  # 0.2 Hz   <=> 5 s

# --- Extraire un segment ---
N = int(0.5 * 3600 * 5)  # 2.5h de données
date_debut = pd.Timestamp("2019-12-19 10:00:00")
date = pd.Timestamp(time_sel)
indice = conv_time(date_debut, date, 1 / fs)

elevation = data["pressure"].values[indice : indice + N]
time_p = data["datetime"].values[indice : indice + N]
dep = np.mean(elevation)
print(f"Profondeur moyenne : {dep:.2f} m")

# --- Spectral analysis ---
freq, E_p = welch(
    elevation,
    fs=fs,
    window="hann",
    nperseg=nfft,
    detrend="linear",
    return_onesided=True,
    scaling="density",
)

# --- Détection du pic dans la bande utile ---
mask_band = (freq >= f_min) & (freq <= f_max)
i_peak = np.argmax(E_p[mask_band])
idx_peak = np.where(mask_band)[0][i_peak]
f_peak = freq[idx_peak]
Tp_peak = 1 / f_peak

print(f"Pic spectral : {f_peak:.3f} Hz → période {Tp_peak:.2f} s")

# --- Plot raw signal ---
plt.figure(figsize=(10, 5))
plt.plot(t[:N], elevation[:N], "m-")
plt.xlabel("Time (s)")
plt.ylabel("Pressure (m of water)")
plt.title("First 1000 samples of raw pressure data")
plt.show()

# --- Plot the full spectrum ---
plt.figure(figsize=(10, 5))
plt.loglog(freq, E_p, "m-", lw=1.8, label="Bottom pressure spectrum")
plt.scatter(
    f_peak,
    E_p[idx_peak],
    color="r",
    s=80,
    zorder=5,
    label=f"Spectral peak Tp={Tp_peak:.1f}s ({f_peak:.3f} Hz)",
)

# Add guides for the useful frequency band
plt.axvline(f_min, color="gray", ls="--", lw=0.8)
plt.axvline(f_max, color="gray", ls="--", lw=0.8)
plt.text(f_min * 1.05, E_p.max() / 3, "25 s", rotation=90, va="center", color="gray")
plt.text(f_max * 1.05, E_p.max() / 3, "5 s", rotation=90, va="center", color="gray")

plt.xlabel("Frequency (Hz)")
plt.ylabel("Spectral density (m²/Hz)")
plt.title("Pressure spectrum — useful period band 5–25 s highlighted")
plt.legend()
plt.grid(True, which="both", ls="--", lw=0.5)
plt.show()